# Qlib 时间维度数据：TSDatasetH + GRU 时间序列模型 (Alpha158Date 版)

**相对于原版的改动**：
- Handler 从 `Alpha158` (158 因子) 升级为 `Alpha158Date` (170 因子)
- 新增 6 个基本面因子：PB, TURNOVER, PE_TTM, TOTAL_MV, VOLUME_RATIO, DV_RATIO
- 新增 6 个日期因子：DAYWEEK, MONTH, QUARTER, DAYMONTH, WEEKYEAR, DAYYEAR
- 模型 d_feat 从 158 改为 170

In [ ]:
import sys
from pathlib import Path

# 确保项目根目录在 sys.path 中
sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name == "daily_quant" else str(Path.cwd()))

import qlib
import pandas as pd
import numpy as np
from qlib.constant import REG_CN
from qlib.utils import init_instance_by_config, flatten_dict
from qlib.workflow import R
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord

# 注册自定义日期算子（通过 qlib.init 的 custom_ops 参数传入）
from daily_quant.ops.date_ops import DayOfWeek, Month, Quarter, DayOfMonth, WeekOfYear, DayOfYear

# 使用自建的 A 股数据
PROVIDER_URI = "C:/Users/pp/.qlib/qlib_data/cn_data_10y"
qlib.init(provider_uri=PROVIDER_URI, region=REG_CN,
          custom_ops=[DayOfWeek, Month, Quarter, DayOfMonth, WeekOfYear, DayOfYear])
print(f"数据路径: {PROVIDER_URI}")

In [20]:
from qlib.data.dataset import DatasetH, TSDatasetH
from qlib.contrib.data.handler import Alpha158

handler_conf = {
    "start_time": "2024-01-01",
    "end_time": "2024-03-31",
    "fit_start_time": "2024-01-01",
    "fit_end_time": "2024-02-29",
    "instruments": "mid_cap",
}

# --- 截面数据 DatasetH ---
ds_flat = DatasetH(
    handler={"class": "Alpha158", "module_path": "qlib.contrib.data.handler", "kwargs": handler_conf},
    segments={"train": ("2024-01-01", "2024-03-31")},
)
flat_sample = ds_flat.prepare("train", col_set="feature")
print(f"DatasetH  (截面): shape={flat_sample.shape} → 每行是一个 (股票, 日期) 的 158 维特征向量")

# --- 时间窗口 TSDatasetH ---
ds_ts = TSDatasetH(
    handler={"class": "Alpha158", "module_path": "qlib.contrib.data.handler", "kwargs": handler_conf},
    segments={"train": ("2024-01-01", "2024-03-31")},
    step_len=30,  # 每个样本包含过去 30 天
)
ts_sampler = ds_ts.prepare("train", col_set="feature")
first_sample = ts_sampler[0]
print(f"\nTSDatasetH (时间窗口): 每个样本 shape={first_sample.shape}")
print(f"  → [30天 时间步, 158 特征] = 过去 30 个交易日 × 158 个因子")
print(f"  → 模型可以看到因子随时间演变的轨迹，而非只看单日快照")

[51872:MainThread](2026-05-30 23:07:30,119) INFO - qlib.timer - [log.py:127] - Time cost: 92.112s | Loading data Done
[51872:MainThread](2026-05-30 23:07:30,197) INFO - qlib.timer - [log.py:127] - Time cost: 0.029s | DropnaLabel Done
[51872:MainThread](2026-05-30 23:07:30,267) INFO - qlib.timer - [log.py:127] - Time cost: 0.067s | CSZScoreNorm Done
[51872:MainThread](2026-05-30 23:07:30,274) INFO - qlib.timer - [log.py:127] - Time cost: 0.151s | fit & process data Done
[51872:MainThread](2026-05-30 23:07:30,276) INFO - qlib.timer - [log.py:127] - Time cost: 92.269s | Init data Done


DatasetH  (截面): shape=(141481, 158) → 每行是一个 (股票, 日期) 的 158 维特征向量


[51872:MainThread](2026-05-30 23:09:00,099) INFO - qlib.timer - [log.py:127] - Time cost: 89.753s | Loading data Done
[51872:MainThread](2026-05-30 23:09:00,197) INFO - qlib.timer - [log.py:127] - Time cost: 0.043s | DropnaLabel Done
[51872:MainThread](2026-05-30 23:09:00,269) INFO - qlib.timer - [log.py:127] - Time cost: 0.068s | CSZScoreNorm Done
[51872:MainThread](2026-05-30 23:09:00,276) INFO - qlib.timer - [log.py:127] - Time cost: 0.175s | fit & process data Done
[51872:MainThread](2026-05-30 23:09:00,280) INFO - qlib.timer - [log.py:127] - Time cost: 89.933s | Init data Done



TSDatasetH (时间窗口): 每个样本 shape=(30, 158)
  → [30天 时间步, 158 特征] = 过去 30 个交易日 × 158 个因子
  → 模型可以看到因子随时间演变的轨迹，而非只看单日快照


In [ ]:
# --- 修复版：用 np.nanmean/nanstd 替代 np.mean/std，避免 NaN 传染 ---
import copy
import numpy as np
from qlib.data.dataset import TSDatasetH, TSDataSampler
from qlib.contrib.data.handler import Alpha158

# ============ 可配置参数 ============
INSTRUMENTS = "mid_cap"     # 股票池: mid_cap, csi300, all, small_cap ...
STEP_LEN = 60               # 时间窗口长度（交易日）
# ====================================

class FixedNormalizedTSDataSampler(TSDataSampler):
    """修复版 TSDataSampler：用 nanmean/nanstd，防止单列 NaN 传染整行"""
    def __getitem__(self, idx):
        data = super().__getitem__(idx)
        process_data = data[:, 0:-1]
        data_mean = np.nanmean(process_data, axis=0)
        data_std = np.nanstd(process_data, axis=0)
        data_std = np.where(data_std < 1e-5, 1.0, data_std)
        normalized = (process_data - data_mean) / data_std
        normalized = np.clip(normalized, -5, 5)
        normalized = np.where(np.isnan(normalized), 0, normalized)
        data[:, 0:-1] = normalized
        return data


class FixedNormalizedTSDatasetH(TSDatasetH):
    def _prepare_seg(self, slc, **kwargs):
        dtype = kwargs.pop("dtype", None)
        if not isinstance(slc, slice):
            slc = slice(*slc)
        flt_col = kwargs.pop("flt_col", None) or self.flt_col
        ext_slice = self._extend_slice(slc, self.cal, self.step_len)
        data = super(TSDatasetH, self)._prepare_seg(ext_slice, **kwargs)
        flt_kwargs = copy.deepcopy(kwargs)
        if flt_col is not None:
            flt_kwargs["col_set"] = flt_col
            flt_data = super(TSDatasetH, self)._prepare_seg(ext_slice, **flt_kwargs)
            assert len(flt_data.columns) == 1
        else:
            flt_data = None
        return FixedNormalizedTSDataSampler(
            data=data, start=slc.start, end=slc.stop,
            step_len=self.step_len, dtype=dtype, flt_data=flt_data,
        )


# 158 (Alpha158) + 6 (基本面) + 6 (日期) = 170 个特征
TOTAL_FEAT = 170

data_handler_config = {
    "start_time": "2022-01-01",
    "end_time": "2026-05-28",
    "fit_start_time": "2022-01-01",
    "fit_end_time": "2024-12-31",
    "instruments": INSTRUMENTS,
}

# 保存 config 字典供后续 R.log_params 使用
dataset_config = {
    "class": "FixedNormalizedTSDatasetH",
    "kwargs": {
        "handler": {"class": "Alpha158Date", "module_path": "daily_quant.handler.alpha158_date", "kwargs": data_handler_config},
        "segments": {
            "train": ("2022-01-01", "2024-12-31"),
            "valid": ("2025-01-01", "2025-06-30"),
            "test": ("2025-01-01", "2026-05-27"),
        },
        "step_len": STEP_LEN,
    },
}

dataset = FixedNormalizedTSDatasetH(
    handler={"class": "Alpha158Date", "module_path": "daily_quant.handler.alpha158_date", "kwargs": data_handler_config},
    segments=dataset_config["kwargs"]["segments"],
    step_len=STEP_LEN,
)

train_ts = dataset.prepare("train", col_set="feature")
print(f"训练样本数: {len(train_ts)}")
print(f"每个样本: {train_ts[0].shape}  → [{STEP_LEN}天, {TOTAL_FEAT}特征]")

In [ ]:
import copy
import torch
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from qlib.contrib.model.pytorch_gru_ts import GRU
from qlib.data.dataset.handler import DataHandlerLP
from qlib.data.dataset.weight import Reweighter
from qlib.model.utils import ConcatDataset
from qlib.utils import get_or_create_path
from qlib.workflow import R
import numpy as np

class GRUWithProgress(GRU):
    """GRU 模型 + tqdm 进度条 + R.log_metrics"""
    def fit(self, dataset, evals_result=dict(), save_path=None, reweighter=None):
        dl_train = dataset.prepare("train", col_set=["feature", "label"], data_key=DataHandlerLP.DK_L)
        dl_valid = dataset.prepare("valid", col_set=["feature", "label"], data_key=DataHandlerLP.DK_L)
        if dl_train.empty or dl_valid.empty:
            raise ValueError("Empty data from dataset, please check your dataset config.")

        dl_train.config(fillna_type="ffill+bfill")
        dl_valid.config(fillna_type="ffill+bfill")

        wl_train = np.ones(len(dl_train)) if reweighter is None else reweighter.reweight(dl_train)
        wl_valid = np.ones(len(dl_valid)) if reweighter is None else reweighter.reweight(dl_valid)

        train_loader = DataLoader(ConcatDataset(dl_train, wl_train),
                                  batch_size=self.batch_size, shuffle=True,
                                  num_workers=self.n_jobs, drop_last=True)
        valid_loader = DataLoader(ConcatDataset(dl_valid, wl_valid),
                                  batch_size=self.batch_size, shuffle=False,
                                  num_workers=self.n_jobs, drop_last=True)

        save_path = get_or_create_path(save_path)
        stop_steps, best_score, best_epoch = 0, -np.inf, 0
        best_param = copy.deepcopy(self.GRU_model.state_dict())
        evals_result["train"], evals_result["valid"] = [], []
        self.logger.info("training...")
        self.fitted = True

        pbar = tqdm(range(self.n_epochs), desc="Training", unit="epoch")
        for step in pbar:
            self.train_epoch(train_loader)
            train_loss, train_score = self.test_epoch(train_loader)
            val_loss, val_score = self.test_epoch(valid_loader)
            evals_result["train"].append(train_score)
            evals_result["valid"].append(val_score)

            R.log_metrics(train_score=train_score, valid_score=val_score, step=step)

            if val_score > best_score:
                best_score, stop_steps, best_epoch = val_score, 0, step
                best_param = copy.deepcopy(self.GRU_model.state_dict())
            else:
                stop_steps += 1
                if stop_steps >= self.early_stop:
                    pbar.set_description(f"Early stop @ epoch {step}")
                    break

            pbar.set_postfix({"train": f"{train_score:.4f}", "valid": f"{val_score:.4f}", "best": f"{best_score:.4f}"})

        self.logger.info("best score: %.6lf @ %d" % (best_score, best_epoch))
        self.GRU_model.load_state_dict(best_param)
        torch.save(best_param, save_path)
        if self.use_gpu:
            torch.cuda.empty_cache()

model_config = {
    "class": "GRUWithProgress",
    "kwargs": {
        "d_feat": TOTAL_FEAT,  # 164 = 158 (Alpha158) + 6 (日期因子)
        "hidden_size": 64,
        "num_layers": 2,
        "dropout": 0.1,
        "n_epochs": 100,
        "lr": 0.001,
        "batch_size": 512,
        "early_stop": 15,
        "loss": "mse",
        "optimizer": "adam",
        "GPU": 0,
        "seed": 42,
        "n_jobs": 0,
    },
}
model = GRUWithProgress(**model_config["kwargs"])

# 动态实验名：模型_股票池_窗口
EXP_NAME = f"GRU_{INSTRUMENTS}_{STEP_LEN}d"
print(f"实验名称: {EXP_NAME}")

with R.start(experiment_name=EXP_NAME):
    R.log_params(**flatten_dict({"model": model_config, "dataset": dataset_config}))
    model.fit(dataset)
    R.save_objects(trained_model=model)
    rid = R.get_recorder().id

    # 自动打标签
    R.set_tags(
        status="completed",
        model_type="GRU",
        instruments=INSTRUMENTS,
        step_len=str(STEP_LEN),
        train_date=pd.Timestamp.now().strftime("%Y-%m-%d"),
    )
    print(f"GRU 训练完成, recorder_id: {rid}")

## 训练完成后

使用 backtest_eval.ipynb 进行回测和评估。

